# 04 - Probar el agente de AgentCore

Este notebook invoca el agente `ecommerceagent` que desplegamos en **AgentCore Runtime**.
El agente tiene dos herramientas:

- `search_products`: consulta el inventario en DynamoDB (marca, categoría, departamento).
- `search_catalog`: consulta la Knowledge Base con las descripciones de productos.

## 1. Configuración

In [ ]:
import json
import uuid

import boto3

REGION = "us-east-1"
# Pega aquí el ARN que te devolvió `agentcore deploy` (o `agentcore status`)
AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:<tu-account-id>:runtime/<tu-runtime-id>"

client = boto3.client("bedrock-agentcore", region_name=REGION)

## 2. Función para invocar el agente

- `payload`: el JSON que recibe el `@app.entrypoint` del agente (`{"prompt": ...}`).
- `runtimeSessionId`: identifica la sesión. Debe tener al menos 33 caracteres (un UUID sirve).

In [ ]:
def ask(prompt, session_id=None):
    session_id = session_id or str(uuid.uuid4())
    response = client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    body = json.loads(response["response"].read())
    return body["response"]

## 3. Prueba: inventario (usa `search_products`)

In [ ]:
print(ask("¿Qué productos de la marca Nike hay en el departamento Men? Dame 3 con su precio."))

## 4. Prueba: catálogo (usa `search_catalog`)

In [ ]:
print(ask("Según el catálogo, ¿qué es el producto Running Shoes Max y de qué marca es?"))

## 5. Prueba: pregunta fuera de alcance

Observa cómo responde el agente a algo que no está en el inventario ni en el catálogo.

In [ ]:
print(ask("¿Cuál es la capital de Francia?"))

## 6. ¿Recuerda la conversación?

Mandamos dos mensajes con el **mismo** `session_id`. Este agente todavía **no tiene memoria**,
así que la segunda pregunta no entiende a qué se refiere "esos".
Agregar memoria es uno de los siguientes pasos para complicar el agente.

In [ ]:
session = str(uuid.uuid4())

print(ask("¿Qué productos Nike hay en el departamento Men? Dame 3.", session))
print("-" * 40)
print(ask("¿Cuál de esos tres es el más barato?", session))